<a href="https://colab.research.google.com/github/MicheleQGF/ProjectAllocationBudgetSynthetic/blob/main/02_ratesEVMandBudgets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resource Allocation & Budget Analysis · Part 4: rates, EVM and Budgets #

In [ ]:
import os
from pathlib import Path

# Este comando encuentra tu carpeta de usuario real, se llame como se llame
user_home = Path.home()
print("Tu carpeta de usuario real es:", user_home)


Tu carpeta de usuario real es: C:\Users\PC


In [ ]:
# Cambia la ruta por la carpeta que tú quieras
nueva_ruta = r"C:\Users\PC\Documents\TAWOS"

os.chdir(nueva_ruta)

# Verificamos que se haya cambiado
print("Ahora estás en:", os.getcwd())

Ahora estás en: C:\Users\PC\Documents\TAWOS


In [ ]:
import pandas as pd, numpy as np

date_cols = ['Creation_Date', 'Estimation_Date', 'Resolution_Date',
             'Last_Updated', 'Sprint_Start', 'Sprint_End']
main = pd.read_csv('tawos_data/main_part3.csv', parse_dates=date_cols)

need = ['Hourly_Rate_USD', 'Rate_Source', 'Owner_Category', 'Assignee_ID_Final', 'Total_Effort_Minutes']
assert not [c for c in need if c not in main.columns], [c for c in need if c not in main.columns]

main['Actual_Hours']    = main['Total_Effort_Minutes'] / 60
main['Actual_Cost_USD'] = main['Actual_Hours'] * main['Hourly_Rate_USD']

# Flag extreme effort (same 30-day threshold used earlier in the notebook)
main['Is_Effort_Outlier'] = main['Total_Effort_Minutes'] > 43200

print("Issues with no rate:", main['Hourly_Rate_USD'].isna().sum(),
      "| no effort recorded:", main['Total_Effort_Minutes'].isna().sum())
print(main.groupby('Owner_Category').agg(
    Issues=('ID', 'count'),
    With_Effort=('Total_Effort_Minutes', lambda x: (x > 0).sum()),
    Hours=('Actual_Hours', 'sum')).round(0))

o = main[main['Is_Effort_Outlier']]
print(len(o), "outliers =", round(o['Total_Effort_Minutes'].sum() / main['Total_Effort_Minutes'].sum() * 100, 1), "% of effort")
print(main.nlargest(10, 'Actual_Cost_USD')[['Issue_Key', 'Status', 'Resolution', 'Owner_Category', 'Actual_Hours', 'Actual_Cost_USD']].round(0))

Issues with no rate: 0 | no effort recorded: 0
                                      Issues  With_Effort      Hours
Owner_Category                                                      
Other contributor (not tiered)           355          123    54861.0
Tiered contributor                     18895         8446  2339013.0
Unassigned - not worked / still open    1471            6    63076.0
Unassigned - worked, owner unknown       133            9    30058.0
509 outliers = 72.0 % of effort
        Issue_Key    Status                Resolution  \
11191   MULE-4530  Resolved         Works as Designed   
2780    TIDOC-288    Closed                      Hold   
9865    MULE-6160    Closed  Won't Fix or Usage Issue   
813    TIDOC-2445    Closed                      Hold   
1514   TIDOC-1679    Closed          Cannot Reproduce   
10005   MULE-5993    Closed  Won't Fix or Usage Issue   
8683    MULE-7706    Closed                  Won't Do   
6808   MULE-10161    Closed                  Won't 

### Flagged: tickets with extreme recorded effort

A handful of tickets carry effort far beyond what a person could have worked. Examples (Mule):
- **MULE-4530** (Enhancement Request): about 74,000 recorded hours over a 3,292-day life.
- **MULE-10161** (Bug, Closed as *Won't Do*): about 27,600 hours.
- **MULE-10113** (Bug, Resolved as *Won't Fix or Usage Issue*): about 18,400 hours.

Two of these three were rejected without a fix, which suggests the effort field is measuring
**how long the ticket stayed open**, not hours of work (see the earlier note on `Total_Effort_Minutes`).

**Decision:** keep them in the dataset and in the cost totals, flagged with `Is_Effort_Outlier`,
so a reader can filter them out. Costs and CPI are shown with them included.

In [ ]:
resolved_mask = main['Status'].isin(['Closed', 'Done', 'Resolved'])   # what the old EVM counted
pd.options.display.float_format = '{:,.2f}'.format

# 1. Cost by project: all issues, resolved only, and resolved without the flagged outliers
by_proj = pd.DataFrame({
    'Cost_All':      main.groupby('Project_Name')['Actual_Cost_USD'].sum(),
    'Cost_Resolved': main[resolved_mask].groupby('Project_Name')['Actual_Cost_USD'].sum(),
    'Cost_Resolved_No_Outliers': main[resolved_mask & ~main['Is_Effort_Outlier']]
                                 .groupby('Project_Name')['Actual_Cost_USD'].sum(),
})
print("Millions of USD")
print((by_proj / 1e6).round(2))
print("\nShare of resolved cost from outlier tickets (%)")
print(((1 - by_proj['Cost_Resolved_No_Outliers'] / by_proj['Cost_Resolved']) * 100).round(1))

# 2. Where does the cost come from? (resolved, % by rate source)
mix = main[resolved_mask].pivot_table(index='Project_Name', columns='Rate_Source',
                                      values='Actual_Cost_USD', aggfunc='sum', fill_value=0)
print("\nCost by rate source (%)")
print((mix.div(mix.sum(axis=1), axis=0) * 100).round(1))

# 3. Preview: old budget rules applied to the new resolved cost
factors = {'Mule': 1.20, 'Titanium Mobile Platform': 0.85}

def preview(col):
    budgets = {}
    for p, actual in by_proj[col].items():
        if p in factors:
            budgets[p] = actual * factors[p]
        else:                                                  # fixed annual, rounded to 100K
            d = main.loc[main['Project_Name'] == p, 'Creation_Date']
            yrs = (d.max() - d.min()).days / 365
            budgets[p] = round((actual / yrs) / 100000) * 100000 * yrs
    b = pd.DataFrame({'Actual': by_proj[col], 'Budget': pd.Series(budgets)})
    b['Variance'] = b['Budget'] - b['Actual']
    b['Variance_%'] = b['Variance'] / b['Budget'] * 100
    b[['Actual', 'Budget', 'Variance']] /= 1e6
    return b.round(2)

print("\nBudget preview, outliers included (millions USD)")
print(preview('Cost_Resolved'))
print("\nBudget preview, outliers excluded (millions USD)")
print(preview('Cost_Resolved_No_Outliers'))

Millions of USD
                          Cost_All  Cost_Resolved  Cost_Resolved_No_Outliers
Project_Name                                                                
Appcelerator Studio          13.93          11.84                       5.82
Mule                         47.45          47.31                      14.81
Titanium Mobile Platform     31.31          31.26                       6.37

Share of resolved cost from outlier tickets (%)
Project_Name
Appcelerator Studio        50.90
Mule                       68.70
Titanium Mobile Platform   79.60
dtype: float64

Cost by rate source (%)
Rate_Source               Assumed (core median)  \
Project_Name                                      
Appcelerator Studio                        1.10   
Mule                                       2.10   
Titanium Mobile Platform                   1.60   

Rate_Source               Assumed (no assignee, core median)  \
Project_Name                                                   
Appcelerator S

In [ ]:
print(main.groupby('Project_Name')['Is_Effort_Outlier'].agg(['sum', 'mean']).round(3))

                          sum  mean
Project_Name                       
Appcelerator Studio        70  0.01
Mule                      257  0.02
Titanium Mobile Platform  182  0.06


In [ ]:
d = main[(main['Total_Effort_Minutes'] > 0) & (main['Resolution_Time_Minutes'] > 0)].copy()
d['Effort_vs_Lifetime'] = d['Total_Effort_Minutes'] / d['Resolution_Time_Minutes']

print(d.groupby('Is_Effort_Outlier')['Effort_vs_Lifetime'].describe().round(2))
print(d.groupby('Project_Name')['Effort_vs_Lifetime'].median().round(2))

# Share of issues where "effort" is within 10% of the ticket's whole life
near = d['Effort_vs_Lifetime'].between(0.9, 1.1)
print("Effort ≈ lifetime:", near.mean().round(3))
print(near.groupby(d['Is_Effort_Outlier']).mean().round(3))
print("Share of issues with effort > lifetime:", (d['Effort_vs_Lifetime'] > 1.1).mean().round(3))

                     count  mean  std  min  25%  50%  75%  max
Is_Effort_Outlier                                             
False             8,037.00  0.29 0.36 0.00 0.01 0.09 0.51 3.10
True                498.00  0.66 0.33 0.02 0.32 0.79 0.99 1.00
Project_Name
Appcelerator Studio        0.04
Mule                       0.15
Titanium Mobile Platform   0.26
Name: Effort_vs_Lifetime, dtype: float64
Effort ≈ lifetime: 0.158
Is_Effort_Outlier
False   0.14
True    0.37
Name: Effort_vs_Lifetime, dtype: float64
Share of issues with effort > lifetime: 0.0


In [ ]:
print((main['In_Progress_Minutes'] > 0).groupby(main['Project_Name']).mean().round(2))
print(main[['In_Progress_Minutes', 'Total_Effort_Minutes']].describe().round(0))

Project_Name
Appcelerator Studio        0.42
Mule                       0.37
Titanium Mobile Platform   0.49
Name: In_Progress_Minutes, dtype: float64
       In_Progress_Minutes  Total_Effort_Minutes
count            20,854.00             20,854.00
mean              6,158.00              7,155.00
std              58,606.00             61,227.00
min                   0.00                  0.00
25%                   0.00                  0.00
50%                   0.00                  0.00
75%                 601.00              1,194.00
max           4,417,718.00          4,417,718.00


In [ ]:
cl = pd.read_csv('tawos_data/Change_Log.csv',
                 usecols=['Issue_ID', 'Change_Type', 'Field', 'From_String', 'To_String'])
tr = cl[(cl['Change_Type'] == 'PEOPLE') & (cl['Field'] == 'assignee')
        & cl['From_String'].notna() & cl['To_String'].notna()]     # same "true reassignment" rule as Part 3

main = main.drop(columns=['Reassign_Count'], errors='ignore')
main = main.merge(tr.groupby('Issue_ID').size().rename('Reassign_Count'),
                  left_on='ID', right_index=True, how='left')
main['Reassign_Count'] = main['Reassign_Count'].fillna(0).astype(int)

print(main.groupby('Is_Effort_Outlier')['Reassign_Count'].describe().round(2))
print(pd.crosstab(main['Is_Effort_Outlier'], main['Reassign_Count'].clip(upper=4),
                  normalize='index').round(3))            # column 4 = "4 or more"

d2 = main[main['Total_Effort_Minutes'] > 0]
print(d2[['Reassign_Count', 'Total_Effort_Minutes', 'Resolution_Time_Minutes', 'Comment_Count']]
      .corr(method='spearman').round(2))                  # rank-based: robust to the huge values

                      count  mean  std  min  25%  50%  75%   max
Is_Effort_Outlier                                               
False             20,345.00  0.47 0.83 0.00 0.00 0.00 1.00 11.00
True                 509.00  0.58 1.01 0.00 0.00 0.00 1.00  7.00
Reassign_Count       0    1    2    3    4
Is_Effort_Outlier                         
False             0.67 0.23 0.07 0.02 0.01
True              0.65 0.21 0.07 0.04 0.02
                         Reassign_Count  Total_Effort_Minutes  \
Reassign_Count                     1.00                 -0.03   
Total_Effort_Minutes              -0.03                  1.00   
Resolution_Time_Minutes            0.21                  0.39   
Comment_Count                      0.31                 -0.03   

                         Resolution_Time_Minutes  Comment_Count  
Reassign_Count                              0.21           0.31  
Total_Effort_Minutes                        0.39          -0.03  
Resolution_Time_Minutes                     

In [ ]:
res = main[main['Status'].isin(['Closed', 'Done', 'Resolved'])].copy()
res['Hours'] = res['Total_Effort_Minutes'] / 60

typical = res.loc[~res['Is_Effort_Outlier'] & (res['Hours'] > 0), 'Hours']
print("Hours on non-flagged tickets with effort:")
print(typical.quantile([.5, .9, .95, .99, .999]).round(1), "\n")

caps = {'p99 of typical tickets': typical.quantile(.99),
        '160 h (about 1 person-month)': 160,
        '720 h (= the current flag, 30 days x 24h)': 720}

out = {'No cap': (res['Hours'] * res['Hourly_Rate_USD']).groupby(res['Project_Name']).sum() / 1e6}
for name, c in caps.items():
    out[name] = (res['Hours'].clip(upper=c) * res['Hourly_Rate_USD']).groupby(res['Project_Name']).sum() / 1e6
print("Resolved cost, millions USD")
print(pd.DataFrame(out).round(2), "\n")

for name, c in caps.items():
    print(f"{name}: cap = {c:,.0f} h, tickets above it = {(res['Hours'] > c).sum()}")

Hours on non-flagged tickets with effort:
0.50    25.70
0.90   253.60
0.95   363.30
0.99   602.20
1.00   698.30
Name: Hours, dtype: float64 

Resolved cost, millions USD
                          No cap  p99 of typical tickets  \
Project_Name                                               
Appcelerator Studio        11.84                    7.35   
Mule                       47.31                   20.69   
Titanium Mobile Platform   31.26                   10.11   

                          160 h (about 1 person-month)  \
Project_Name                                             
Appcelerator Studio                               4.59   
Mule                                             11.89   
Titanium Mobile Platform                          4.85   

                          720 h (= the current flag, 30 days x 24h)  
Project_Name                                                         
Appcelerator Studio                                            7.71  
Mule                        

In [ ]:
CAP_HOURS = 720
main['Capped_Hours']    = main['Actual_Hours'].clip(upper=CAP_HOURS)
main['Capped_Cost_USD'] = main['Capped_Hours'] * main['Hourly_Rate_USD']

chk = main[main['Status'].isin(['Closed', 'Done', 'Resolved'])]
print((chk.groupby('Project_Name')[['Actual_Cost_USD', 'Capped_Cost_USD']].sum() / 1e6).round(2))
print("Tickets capped:", (main['Actual_Hours'] > CAP_HOURS).sum(), "| flagged:", main['Is_Effort_Outlier'].sum())

                          Actual_Cost_USD  Capped_Cost_USD
Project_Name                                              
Appcelerator Studio                 11.84             7.71
Mule                                47.31            21.95
Titanium Mobile Platform            31.26            10.89
Tickets capped: 509 | flagged: 509


### Flagged: tickets with extreme recorded effort

509 tickets (70 in Appcelerator, 257 in Mule, 182 in Titanium) have more than 30 days of recorded
effort. They are 1-6% of each project's issues, but they make up 51% (Appcelerator), 69% (Mule) and
80% (Titanium) of resolved cost. Appearing in all three projects points to how effort is **recorded**,
not to unusual tickets.

**Evidence that the field tracks elapsed time**
- For the flagged tickets, effort is a median of **79%** of the ticket's whole life (99% at the 75th percentile), and never above 100%. For other tickets the median is 9%.
- Effort correlates 0.39 with how long a ticket was open and about 0 with the number of comments or reassignments.
- `In_Progress_Minutes` is the main component of `Total_Effort_Minutes` (mean 6,158 vs 7,155), so it is not an independent check.
- Reassignments do **not** explain it: 65% of flagged tickets were never reassigned, against 67% of the rest.

**Interpretation.** A likely cause is tickets left in "In Progress" while the clock kept running.
This is not confirmed, because status history was not examined.

**Decision: cap the hours at 720 per ticket, and keep the raw values.**
I first decided to keep these hours as the best available proxy for effort. The evidence above
changed that: since most of the recorded effort on these tickets is elapsed time, using it at face
value would let a few hundred tickets decide each project's cost and budget.

- **Why 720 hours:** it equals the 30 days (30 × 24 h) already used to flag these tickets, so the cap touches exactly the flagged tickets (498 of the resolved ones) and no ordinary ones. Two alternatives were compared: a cap at the 99th percentile of ordinary tickets (602 h) gave almost the same result but cut 81 more tickets, and a cap of 160 h (about one person-month) would have cut 2,001 tickets, including many ordinary ones.
- **Nothing is deleted.** The original `Total_Effort_Minutes` and `Actual_Cost_USD` stay in the dataset. The capped versions are separate columns (`Capped_Hours`, `Capped_Cost_USD`), and `Is_Effort_Outlier` still marks the tickets.
- **The capped version is the main one** for costs, planned hours and budgets. The uncapped version is reported alongside it as an upper bound.
- The cap is applied **before** planned hours are calculated, so planned and actual hours are on the same basis.

| Resolved cost (millions USD) | Uncapped | Capped at 720 h |
|---|---|---|
| Appcelerator Studio | 11.84 | 7.71 |
| Mule | 47.31 | 21.95 |
| Titanium Mobile Platform | 31.26 | 10.89 |
| **Total** | **90.41** | **40.55** |

**Limits**
- The cap is a judgment call. It understates the work on tickets that really were large.
- It is a ceiling, not a correction: each capped ticket is still counted at 720 hours, which is
  about a third of the capped total (roughly $13.6M of $40.55M). The true effort on those tickets
  is unknown.
- Costs are attributed to the final owner, even though about a third of all tickets were reassigned
  at least once.

- It is a ceiling, not a correction: each capped ticket is still counted at 720 hours, and these
  tickets are still $13.55M (33%) of the $40.55M capped total. The true effort on them is unknown.

In [ ]:
chk = main[main['Status'].isin(['Closed', 'Done', 'Resolved'])]
x = chk.loc[chk['Is_Effort_Outlier'], 'Capped_Cost_USD'].sum() / 1e6
print(round(x, 2), "M |", round(x / (chk['Capped_Cost_USD'].sum() / 1e6) * 100, 1), "% of capped total")

13.55 M | 33.4 % of capped total


#

# Step 3 and 4: planned hours, planned value and monthly EVM #

In [ ]:
cl = pd.read_csv('tawos_data/Change_Log.csv',
                 usecols=['Issue_ID', 'Change_Type', 'From_String', 'To_String', 'Creation_Date'],
                 parse_dates=['Creation_Date'])
sp = cl[cl['Change_Type'] == 'STORY_POINT'].copy()
sp['Delta'] = (pd.to_numeric(sp['To_String'], errors='coerce').fillna(0)      # a blank "from" is the first estimate;
               - pd.to_numeric(sp['From_String'], errors='coerce').fillna(0))  # a blank "to" removes the points
sp['Month'] = sp['Creation_Date'].dt.to_period('M').dt.to_timestamp()
sp = sp.merge(main[['ID', 'Project_Name']], left_on='Issue_ID', right_on='ID')

# Issues that never changed: their points enter at the estimation month
main['Estimation_Month'] = main['Estimation_Date'].dt.to_period('M').dt.to_timestamp()
never = main[main['Story_Point'].notna() & main['Estimation_Date'].notna() & ~main['ID'].isin(sp['Issue_ID'])]

flow = pd.concat([never.groupby(['Project_Name', 'Estimation_Month'])['Story_Point'].sum().rename_axis(['Project_Name', 'Month']),
                  sp.groupby(['Project_Name', 'Month'])['Delta'].sum()])
scope = flow.groupby(level=[0, 1]).sum().rename('Scope_Change').reset_index().sort_values(['Project_Name', 'Month'])
scope['Scope_Points'] = scope.groupby('Project_Name')['Scope_Change'].cumsum()

print(scope.groupby(['Project_Name', scope['Month'].dt.year])['Scope_Points'].last().round(0).unstack(0))
chk = pd.DataFrame({'From_log': scope.groupby('Project_Name')['Scope_Points'].last(),
                    'Current_points': main.groupby('Project_Name')['Story_Point'].sum()})
chk['Diff_%'] = ((chk['From_log'] / chk['Current_points'] - 1) * 100).round(1)
print(chk.round(0))

# Can sprints serve as the schedule?
p = main[main['Story_Point'].notna()]
print("\nPointed issues with a sprint:\n", p['Sprint_ID'].notna().groupby(p['Project_Name']).mean().round(2))
print("Sprints per project:\n", main.groupby('Project_Name')['Sprint_ID'].nunique())
print("Resolution date minus sprint end (days):\n",
      (p['Resolution_Date'] - p['Sprint_End']).dt.days.describe().round(0))

Project_Name  Appcelerator Studio      Mule  Titanium Mobile Platform
Month                                                                
2011                       382.00       NaN                       NaN
2012                     6,549.00       NaN                     48.00
2013                    10,724.00    560.00                       NaN
2014                    13,286.00  2,666.00                  1,550.00
2015                    14,673.00  3,784.00                  2,626.00
2016                    15,478.00  8,068.00                  3,224.00
2017                    16,473.00 15,746.00                  4,498.00
2018                    16,970.00 16,913.00                  5,016.00
2019                    17,050.00 16,909.00                  5,502.00
2020                    17,119.00 16,909.00                  5,506.00
                          From_log  Current_points  Diff_%
Project_Name                                              
Appcelerator Studio      17,119.00       1

In [ ]:

RESOLVED  = ['Closed', 'Done', 'Resolved']
COMPLETED = ['Fixed', 'Done', 'Completed']
SP_COL = 'Story_Point'
K = 3             # previous sprints that set the baseline
MIN_ISSUES = 5    # a sprint needs at least this many pointed issues

p = main[main[SP_COL].notna() & main['Sprint_ID'].notna() & main['Sprint_End'].notna()].copy()
p['Finished']  = p['Status'].isin(RESOLVED)
p['Completed'] = p['Finished'] & p['Resolution'].isin(COMPLETED)
p['Late']      = p['Finished'] & (p['Resolution_Date'] > p['Sprint_End'])
p['EV_Pts']    = p[SP_COL].where(p['Completed'], 0)
p['AC_Hours']  = p['Capped_Hours'].where(p['Finished'], 0)

sp = (p.groupby(['Project_Name', 'Sprint_ID'])
        .agg(Sprint_End=('Sprint_End', 'first'), Issues=('ID', 'count'),
             Scope_Pts=(SP_COL, 'sum'), EV_Pts=('EV_Pts', 'sum'), AC_Hours=('AC_Hours', 'sum'),
             Finished=('Finished', 'sum'), Late=('Late', 'sum'))
        .reset_index().sort_values(['Project_Name', 'Sprint_End']).reset_index(drop=True))

# Baseline: hours per completed point over the PREVIOUS K sprints (never the sprint itself)
frames = []
for proj, g in sp.groupby('Project_Name'):
    g = g.copy()
    g['Baseline_HPP'] = (g['AC_Hours'].shift(1).rolling(K, min_periods=2).sum()
                         / g['EV_Pts'].shift(1).rolling(K, min_periods=2).sum())
    frames.append(g)
sp = pd.concat(frames).replace([np.inf, -np.inf], np.nan)

sp['Actual_HPP'] = sp['AC_Hours'] / sp['EV_Pts']
sp['PV_Hours']   = sp['Scope_Pts'] * sp['Baseline_HPP']
sp['EV_Hours']   = sp['EV_Pts']    * sp['Baseline_HPP']
sp['Cum_Scope_Pts'] = sp.groupby('Project_Name')['Scope_Pts'].cumsum()
sp['BAC_Hours']  = sp['Cum_Scope_Pts'] * sp['Baseline_HPP']

sp['Sprint_Status'] = np.select([sp['Baseline_HPP'].isna(), sp['Issues'] < MIN_ISSUES],
                                ['No baseline yet', 'Too few issues'], default='Measurable')
m = sp['Sprint_Status'] == 'Measurable'
for c in ['PV_Hours', 'EV_Hours', 'AC_Hours']:
    sp[f'Cum_{c}'] = sp[c].where(m, 0).groupby(sp['Project_Name']).cumsum()

sp['SPI'] = (sp['Cum_EV_Hours'] / sp['Cum_PV_Hours']).where(m)
sp['CPI'] = (sp['Cum_EV_Hours'] / sp['Cum_AC_Hours']).where(m)
sp = sp.replace([np.inf, -np.inf], np.nan)
sp['Year'] = sp['Sprint_End'].dt.year
sp.to_csv('tawos_data/evm_sprint.csv', index=False)

# --- Is there enough sprint data? ---
pointed = main[main[SP_COL].notna()]
print("Share of pointed issues that have a sprint:\n", pointed['Sprint_ID'].notna().groupby(pointed['Project_Name']).mean().round(2), "\n")
print(sp.groupby(['Project_Name', 'Sprint_Status']).size().unstack(fill_value=0), "\n")
print("Finished issues resolved after their sprint ended:\n", (sp.groupby('Project_Name')['Late'].sum() / sp.groupby('Project_Name')['Finished'].sum()).round(2), "\n")
ok = sp[m]
print(ok.groupby(['Project_Name', 'Year'])[['Scope_Pts', 'BAC_Hours', 'Cum_PV_Hours', 'Cum_EV_Hours', 'Cum_AC_Hours', 'SPI', 'CPI']].last().round(2))

Share of pointed issues that have a sprint:
 Project_Name
Appcelerator Studio        0.38
Mule                       0.81
Titanium Mobile Platform   0.76
Name: Sprint_ID, dtype: float64 

Sprint_Status             Measurable  No baseline yet  Too few issues
Project_Name                                                         
Appcelerator Studio               87                2              55
Mule                             167                2              74
Titanium Mobile Platform          74                2              86 

Finished issues resolved after their sprint ended:
 Project_Name
Appcelerator Studio        0.20
Mule                       0.34
Titanium Mobile Platform   0.26
dtype: float64 

                               Scope_Pts  BAC_Hours  Cum_PV_Hours  \
Project_Name             Year                                       
Appcelerator Studio      2013      90.00   5,472.13      7,670.69   
                         2014      37.00  42,195.53     29,610.71   
      

In [ ]:
chg = cl[cl['Change_Type'] == 'STORY_POINT'].sort_values('Creation_Date').copy()
chg['From_N'] = pd.to_numeric(chg['From_String'], errors='coerce')
chg['To_N']   = pd.to_numeric(chg['To_String'],   errors='coerce')
chg['Delta']  = chg['To_N'].fillna(0) - chg['From_N'].fillna(0)
chg['Month']  = chg['Creation_Date'].dt.to_period('M').dt.to_timestamp()
chg = chg.merge(main[['ID', 'Project_Name']], left_on='Issue_ID', right_on='ID')

# Opening estimate: the value the first logged change starts from
first = chg.groupby('Issue_ID').head(1)
init = (first[first['From_N'].fillna(0) > 0]
        .merge(main[['ID', 'Estimation_Month']], on='ID'))
init['Month'] = init['Estimation_Month'].fillna(init['Month'])
print("Opening points never counted before:", init['From_N'].sum(), "| issues:", len(init))

never = main[main['Story_Point'].notna() & main['Estimation_Date'].notna()
             & ~main['ID'].isin(chg['Issue_ID'])]
flows = [never.groupby(['Project_Name', 'Estimation_Month'])['Story_Point'].sum().rename_axis(['Project_Name', 'Month']),
         chg.groupby(['Project_Name', 'Month'])['Delta'].sum(),
         init.groupby(['Project_Name', 'Month'])['From_N'].sum()]
scope = (pd.concat(flows).groupby(level=[0, 1]).sum().rename('Scope_Change')
           .reset_index().sort_values(['Project_Name', 'Month']))
scope['Scope_Points'] = scope.groupby('Project_Name')['Scope_Change'].cumsum()

chk = pd.DataFrame({'From_log': scope.groupby('Project_Name')['Scope_Points'].last(),
                    'Current_points': main.groupby('Project_Name')['Story_Point'].sum()})
chk['Diff_%'] = ((chk['From_log'] / chk['Current_points'] - 1) * 100).round(1)
print(chk.round(0))

last = chg.groupby('Issue_ID').tail(1).set_index('Issue_ID')['To_N']
cur = main.set_index('ID')['Story_Point']
print("Last logged value equals current points:", (last.fillna(0) == cur.reindex(last.index).fillna(0)).mean().round(3))

Opening points never counted before: 3107.0 | issues: 405
                          From_log  Current_points  Diff_%
Project_Name                                              
Appcelerator Studio      18,127.00       18,658.00   -3.00
Mule                     18,598.00       18,593.00    0.00
Titanium Mobile Platform  5,916.00        5,916.00    0.00
Last logged value equals current points: 0.992


In [ ]:
bad = pd.DataFrame({'Last_Logged': last, 'Current': cur.reindex(last.index)}).fillna(0)
bad = bad[bad['Last_Logged'] != bad['Current']].join(main.set_index('ID')[['Project_Name', 'Issue_Key']])
bad['Diff'] = bad['Current'] - bad['Last_Logged']
print(bad.groupby('Project_Name')['Diff'].agg(['count', 'sum']))
print(bad.sort_values('Diff', key=abs, ascending=False).head(8))

                     count    sum
Project_Name                     
Appcelerator Studio     47 386.00
Mule                     2  -5.00
          Last_Logged  Current         Project_Name    Issue_Key  Diff
Issue_ID                                                              
401280           0.00    20.00  Appcelerator Studio  TISTUD-1581 20.00
401877           0.00    20.00  Appcelerator Studio   TISTUD-794 20.00
400498           0.00    20.00  Appcelerator Studio  TISTUD-2646 20.00
401674           0.00    13.00  Appcelerator Studio  TISTUD-1023 13.00
401861           0.00    13.00  Appcelerator Studio   TISTUD-811 13.00
401778           0.00    13.00  Appcelerator Studio   TISTUD-907 13.00
402183           0.00    13.00  Appcelerator Studio   TISTUD-434 13.00
402464           0.00    13.00  Appcelerator Studio   TISTUD-121 13.00


In [ ]:
sprints = main[['Sprint_ID', 'Sprint_Start', 'Sprint_End']].drop_duplicates('Sprint_ID')
c = chg.merge(main[['ID', 'Sprint_ID']], left_on='Issue_ID', right_on='ID').merge(sprints, on='Sprint_ID')
c['Added_Mid_Sprint'] = (c['Creation_Date'] > c['Sprint_Start']) & (c['Creation_Date'] <= c['Sprint_End'])
c['Opening'] = c['Issue_ID'].isin(init['Issue_ID']) & (c['Creation_Date'] == c.groupby('Issue_ID')['Creation_Date'].transform('min'))

mid = (c[c['Added_Mid_Sprint'] & ~c['Opening']]
       .groupby(['Project_Name', 'Sprint_ID'])['Delta'].agg(Net_Change='sum', Changes='count').reset_index())
base = sp.groupby(['Project_Name', 'Sprint_ID'])['Scope_Pts'].sum().reset_index()
mid = mid.merge(base, on=['Project_Name', 'Sprint_ID'])
mid['Pct_of_Scope'] = mid['Net_Change'] / mid['Scope_Pts'] * 100
print(mid.groupby('Project_Name')[['Net_Change', 'Pct_of_Scope']].describe().T.round(1))

Project_Name        Appcelerator Studio    Mule  Titanium Mobile Platform
Net_Change   count                92.00  197.00                     84.00
             mean                 22.20   16.60                      9.50
             std                  30.60   22.40                     14.30
             min                  -3.00  -70.00                    -32.00
             25%                   5.00    3.00                      2.00
             50%                  10.00   11.00                      4.00
             75%                  26.00   25.00                     15.00
             max                 177.00  106.00                     67.00
Pct_of_Scope count                91.00  195.00                     84.00
             mean                 40.30   32.60                     30.00
             std                  29.30   36.20                     26.40
             min                  -7.30 -100.00                    -20.00
             25%                  16.4

In [ ]:
COMPLETED = ['Fixed', 'Done', 'Completed']
REJECTED  = ['Abandoned', 'Cannot Reproduce', 'Duplicate', 'Incomplete', 'Invalid', 'Not Our Bug',
             'Rejected', "Won't Do", "Won't Fix", "Won't Fix or Usage Issue",
             'Works as Designed', 'Needs more info']          # 'Hold' stays open

pt = main[main['Story_Point'].notna()].copy()
pt['Outcome'] = np.select([pt['Resolution'].isin(COMPLETED), pt['Resolution'].isin(REJECTED)],
                          ['Completed', 'Rejected'], default='Open / other')
out = pt.groupby(['Project_Name', 'Outcome']).agg(
    Issues=('ID', 'count'), Points=('Story_Point', 'sum'), Hours=('Capped_Hours', 'sum'))
out['Pct_of_Points'] = (out['Points'] / out.groupby(level=0)['Points'].transform('sum') * 100).round(1)
print(out.round(0))

rj = pt[pt['Outcome'] == 'Rejected'].copy()
print((rj['Resolution_Date'] - rj['Sprint_End']).dt.days.groupby(rj['Project_Name']).describe().round(0))
print(rj['Resolution_Date'].dt.date.value_counts().head(5))     # bulk closures show up here

                                       Issues    Points      Hours  \
Project_Name             Outcome                                     
Appcelerator Studio      Completed       2742 14,880.00 150,894.00   
                         Open / other     188  1,221.00   9,635.00   
                         Rejected         476  2,557.00  15,394.00   
Mule                     Completed       3459 14,456.00 282,220.00   
                         Open / other     208  1,645.00   1,782.00   
                         Rejected         503  2,492.00  14,669.00   
Titanium Mobile Platform Completed       1178  5,193.00 154,050.00   
                         Open / other      16    163.00   1,371.00   
                         Rejected         103    560.00   9,112.00   

                                       Pct_of_Points  
Project_Name             Outcome                      
Appcelerator Studio      Completed             80.00  
                         Open / other           6.00  
         

### Rejected work: scope is removed when the decision is made

Issues closed without being done (*Won't Fix*, *Duplicate*, *Cannot Reproduce*, *Won't Do* and similar) hold
14% (Appcelerator), 13% (Mule) and 10% (Titanium) of all story points, and used 15,394, 14,669 and 9,112 hours.

**Treatment**
- Their points count in the plan when they are estimated.
- On the resolution date, the same points are **removed from the scope** (a negative entry). The scope shows
  what the team was still committed to.
- They earn no value, since no work product was delivered.
- Their hours **stay in the actual cost**, because the effort was spent. The waste appears in cost efficiency.

**Why.** If rejected points stayed in the plan forever, the schedule index could never reach 1, and this would be
blamed on slow delivery when it is really cancelled work. Removing them on the day of the decision keeps the
schedule index about delivery and leaves the cost of waste visible.

**Limits**
- The descope is booked on the resolution date. About a quarter of the rejections of Mule and Titanium happen
  two or more weeks after the sprint ended, so in the sprint where the issue was planned it still looks unfinished.
- Some days hold many rejections (for example 40 issues on 2020-07-27), which moves the scope of that month a lot.
- Both versions are saved: `SPI` (with descope) and `SPI_No_Descope`.

In [ ]:
pt = main[main['Story_Point'].notna() & main['Status'].isin(['Closed', 'Done', 'Resolved'])].copy()
pt['Has_Effort'] = pt['Total_Effort_Minutes'] > 0
t = pt.groupby('Resolution').agg(Issues=('ID', 'count'),
                                 Pct_With_Effort=('Has_Effort', 'mean'),
                                 Median_Hours_If_Any=('Capped_Hours', lambda s: s[s > 0].median()),
                                 Pct_Reassigned=('Was_Reassigned', 'mean'),
                                 Median_Comments=('Comment_Count', 'median'))
t['Pct_With_Effort'] = (t['Pct_With_Effort'] * 100).round(0)
t['Pct_Reassigned'] = (t['Pct_Reassigned'] * 100).round(0)
print(t.sort_values('Issues', ascending=False).round(1))

                          Issues  Pct_With_Effort  Median_Hours_If_Any  \
Resolution                                                               
Fixed                       5851            75.00                24.80   
Completed                    944            68.00                89.80   
Done                         584            81.00                52.20   
Won't Fix or Usage Issue     231            30.00                22.60   
Duplicate                    221            22.00                37.60   
Cannot Reproduce             167            45.00                19.00   
Won't Fix                    145            23.00                34.10   
Won't Do                     129             9.00               142.20   
Invalid                      101            30.00                 2.90   
Not Our Bug                   30            47.00                 1.80   
Works as Designed             22            55.00                25.90   
Needs more info               18      

### Issues closed without delivery: scope is removed when they are closed

Issues closed with a resolution such as *Won't Fix*, *Duplicate*, *Cannot Reproduce* or *Won't Do* are called
**closed without delivery**. They hold 14% (Appcelerator), 13% (Mule) and 10% (Titanium) of all story points.
The label records the outcome, **not whether work was done first**, so I don't call them waste.

**What the data shows**
- Roughly 28% of them have any recorded effort, against roughly 75% of completed issues (Fixed, Completed, Done).
  Most were therefore closed before effort was logged.
- Some labels look like quick closures: *Won't Fix or Usage Issue* and *Won't Do* have few handoffs (10-12%) and about one comment.
  Several closed in bulk on the same day (for example 40 issues on 2020-07-27).
- Others look investigated: *Cannot Reproduce*, *Not Our Bug* and *Works as Designed* have more handoffs (45-67%) and discussion.
- When effort is recorded, its median is 19-142 hours, similar to completed issues. The data cannot tell real work
  from a clock that kept running.

**Treatment**
- Points count in the plan when the issue is estimated.
- On the resolution date the same points are **removed from the scope**, because they never became delivered work.
- They earn no value.
- Their recorded hours **stay in the actual cost**, since the data does not show whether the effort was needed.

**Why.** If these points stayed in the plan forever, the schedule index could never reach 1, and that would look like
slow delivery when it is closed work. Removing them on the day of the decision keeps the schedule index about
delivery, and the cost side stays neutral.

**Limits**
- Hours on these issues may be real triage (likely for *Cannot Reproduce*) or elapsed time. This is a limit of the data.
- For *Duplicate*, the work may have been done under another issue. This was not checked.
- The descope is booked on the resolution date. About a quarter of Mule's and Titanium's closures come two or more weeks after the sprint ended, so in the sprint where they were planned they still look unfinished.
- Both versions are saved: `SPI` (with descope) and `SPI_No_Descope`.

In [ ]:
# ---------- 0. Settings ----------
RESOLVED  = ['Closed', 'Done', 'Resolved']
COMPLETED = ['Fixed', 'Done', 'Completed']
CLOSED_WITHOUT_DELIVERY = ['Abandoned', 'Cannot Reproduce', 'Duplicate', 'Incomplete', 'Invalid',
                           'Not Our Bug', 'Rejected', "Won't Do", "Won't Fix",
                           "Won't Fix or Usage Issue", 'Works as Designed', 'Needs more info']
MIN_BASE_PTS = 100            # completed points needed before a baseline is used
MIN_TIME = 0.05               # SPI is not shown in the first 5% of a project's measured timeline (tiny divisor)
END_PCT = 1.0                 # plan ends when this share of the final earned points is first reached
ns = 'datetime64[ns]'

# ---------- 1. Story-point history: every change with its date ----------
cl = pd.read_csv('tawos_data/Change_Log.csv',
                 usecols=['Issue_ID', 'Change_Type', 'From_String', 'To_String', 'Creation_Date'],
                 parse_dates=['Creation_Date'])
chg = cl[cl['Change_Type'] == 'STORY_POINT'].sort_values('Creation_Date').copy()
chg['From_N'] = pd.to_numeric(chg['From_String'], errors='coerce')
chg['To_N']   = pd.to_numeric(chg['To_String'],   errors='coerce')
chg['Creation_Date'] = chg['Creation_Date'].astype(ns)
chg['Issue_ID'] = chg['Issue_ID'].astype('int64')

est = main[['ID', 'Estimation_Date']].copy()
est['Estimation_Date'] = est['Estimation_Date'].astype(ns)

ev_change = pd.DataFrame({'ID': chg['Issue_ID'], 'Date': chg['Creation_Date'],
                          'Pts': chg['To_N'].fillna(0)})          # blank "to" = points removed

first = chg.groupby('Issue_ID').head(1).merge(est, left_on='Issue_ID', right_on='ID', how='left')
first = first[first['From_N'].fillna(0) > 0]                      # opening estimate = first "from"
use_est = first['Estimation_Date'].notna() & (first['Estimation_Date'] <= first['Creation_Date'])
ev_open = pd.DataFrame({'ID': first['Issue_ID'],
                        'Date': first['Estimation_Date'].where(use_est, first['Creation_Date']),
                        'Pts': first['From_N']})

pointed = main[main['Story_Point'].notna()]
never = pointed[pointed['Estimation_Date'].notna() & ~pointed['ID'].isin(chg['Issue_ID'])]
ev_never = pd.DataFrame({'ID': never['ID'], 'Date': never['Estimation_Date'].astype(ns),
                         'Pts': never['Story_Point']})            # never changed: points from estimation date

events_base = (pd.concat([ev_change, ev_open, ev_never])
                 .astype({'ID': 'int64', 'Date': ns}).sort_values('Date'))

# ---------- 2. Issues that belong to a sprint ----------
iss = main[main['Story_Point'].notna() & main['Sprint_ID'].notna()
           & main['Sprint_Start'].notna() & main['Sprint_End'].notna()][
    ['ID', 'Project_Name', 'Sprint_ID', 'Sprint_Start', 'Sprint_End',
     'Status', 'Resolution', 'Resolution_Date', 'Capped_Hours']].copy()
for c in ['Sprint_Start', 'Sprint_End', 'Resolution_Date']:
    iss[c] = iss[c].astype(ns)
iss['ID'] = iss['ID'].astype('int64')
iss['Finished'] = iss['Status'].isin(RESOLVED) & iss['Resolution_Date'].notna()
iss['Completed'] = iss['Finished'] & iss['Resolution'].isin(COMPLETED)
iss['Closed_No_Delivery'] = iss['Finished'] & iss['Resolution'].isin(CLOSED_WITHOUT_DELIVERY)

# Descope: points leave the scope (1 second after) the moment the issue is closed without delivery
cd = iss[iss['Closed_No_Delivery']]
ev_desc = pd.DataFrame({'ID': cd['ID'], 'Date': cd['Resolution_Date'] + pd.Timedelta(seconds=1), 'Pts': 0.0})
events_desc = pd.concat([events_base, ev_desc]).sort_values('Date')

def scope_at(events, df, date_col):
    """Story points each issue held at the given date (last logged value before it)."""
    left = df[['ID', date_col]].dropna().sort_values(date_col)
    m = pd.merge_asof(left, events, left_on=date_col, right_on='Date', by='ID', direction='backward')
    return df['ID'].map(m.set_index('ID')['Pts']).fillna(0)

# ---------- 3. Scope at each sprint's start and end ----------
iss['Pts_Start']  = scope_at(events_desc, iss, 'Sprint_Start')
iss['Pts_End']    = scope_at(events_desc, iss, 'Sprint_End')       # with descope
iss['Pts_End_ND'] = scope_at(events_base, iss, 'Sprint_End')       # ND = no descope

sp = (iss.groupby(['Project_Name', 'Sprint_ID'])
        .agg(Sprint_Start=('Sprint_Start', 'first'), Sprint_End=('Sprint_End', 'first'),
             Issues=('ID', 'count'), Scope_Start_Pts=('Pts_Start', 'sum'),
             Scope_End_Pts=('Pts_End', 'sum'), Scope_End_Pts_ND=('Pts_End_ND', 'sum'))
        .reset_index().sort_values(['Project_Name', 'Sprint_End', 'Sprint_Start']).reset_index(drop=True))

# ---------- 4. Book earned points and hours in the sprint whose end follows the resolution date ----------
def attach_row(df, date_col):
    row = pd.Series(np.nan, index=df.index)
    for p, g in df.groupby('Project_Name'):
        sub = sp[sp['Project_Name'] == p]
        ends = sub['Sprint_End'].values
        pos = np.searchsorted(ends, g[date_col].values, side='left')
        ok = pos < len(ends)                                       # later than the last sprint: not counted
        row.loc[g.index[ok]] = sub.index.values[pos[ok]]
    return row

def by_row(df, col):
    d = df.dropna(subset=['Row'])
    return d.groupby(d['Row'].astype(int))[col].sum().reindex(sp.index).fillna(0)

comp = iss[iss['Completed']].copy()
comp['Pts_Res'] = scope_at(events_base, comp, 'Resolution_Date')   # points as of the day it was resolved
comp['Row'] = attach_row(comp, 'Resolution_Date')
fin = iss[iss['Finished']].copy()
fin['Row'] = attach_row(fin, 'Resolution_Date')
late = iss[iss['Closed_No_Delivery'] & (iss['Resolution_Date'] > iss['Sprint_End'])].copy()
late['Pts_Before'] = scope_at(events_base, late, 'Resolution_Date')
late['Row'] = attach_row(late, 'Resolution_Date')

sp['EV_Pts']           = by_row(comp, 'Pts_Res')            # points completed
sp['AC_Done_Hours']    = by_row(comp, 'Capped_Hours')       # hours of the completed issues
sp['AC_Hours']         = by_row(fin,  'Capped_Hours')       # hours of every finished issue
sp['Late_Descope_Pts'] = by_row(late, 'Pts_Before')         # descoped after their own sprint ended
print("Finished after the project's last sprint (not counted):", int(fin['Row'].isna().sum()), "of", len(fin))

# ---------- 5. Baseline: hours per completed point over ALL previous sprints, new value every sprint ----------
g = sp.groupby('Project_Name')
sp['Prev_Done_Hours'] = g['AC_Done_Hours'].transform(lambda s: s.cumsum().shift(1))
sp['Prev_EV_Pts']     = g['EV_Pts'].transform(lambda s: s.cumsum().shift(1))
sp['Baseline_HPP'] = (sp['Prev_Done_Hours'] / sp['Prev_EV_Pts']).where(sp['Prev_EV_Pts'] >= MIN_BASE_PTS)
sp['Actual_HPP']   = (sp['AC_Done_Hours'] / sp['EV_Pts']).replace([np.inf, -np.inf], np.nan)   # that sprint alone
sp['Priced'] = sp['Baseline_HPP'].notna()
sp['Sprint_Status'] = np.where(sp['Priced'], 'Measurable', 'No baseline yet')

# ---------- 6. EV, AC, BAC, CPI (unchanged) and the time-based PV and SPI ----------
P = sp['Priced']
def cum(s):                                                  # running total per project, from the first priced sprint
    return s.where(P, 0).groupby(sp['Project_Name']).cumsum()

sp['EV_Hours'] = sp['EV_Pts'] * sp['Baseline_HPP']
sp['Cum_EV_Hours'] = cum(sp['EV_Hours'])
sp['Cum_AC_Hours'] = cum(sp['AC_Hours'])
sp['Cum_EV_Pts']   = cum(sp['EV_Pts'])
sp['Cum_BAC_Pts']    = cum(sp['Scope_End_Pts'] - sp['Late_Descope_Pts'])   # scope at the end, descopes removed
sp['Cum_BAC_Pts_ND'] = cum(sp['Scope_End_Pts_ND'])                         # same without descope
sp['BAC_Hours']    = (sp['Cum_BAC_Pts'] * sp['Baseline_HPP']).where(P)
sp['BAC_Hours_ND'] = (sp['Cum_BAC_Pts_ND'] * sp['Baseline_HPP']).where(P)

# Time-based plan: the points delivered in total are spread evenly from the first priced sprint
# to the sprint where the last earned point was delivered (END_PCT = 1.0 -> 100 % of earned points)
t0 = sp[P].groupby('Project_Name')['Sprint_Start'].min()       # measured period starts at the first priced sprint
sp['Final_EV_Pts'] = sp.groupby('Project_Name')['Cum_EV_Pts'].transform('max')   # points delivered in total
reached = sp[P & (sp['Cum_EV_Pts'] >= sp['Final_EV_Pts'] * END_PCT - 1e-9)]
t1 = reached.groupby('Project_Name')['Sprint_End'].min()       # plan ends when the final total is first reached
start = sp['Project_Name'].map(t0)
end   = sp['Project_Name'].map(t1)
sp['Pct_Time']     = ((sp['Sprint_End'] - start) / (end - start)).clip(0, 1)
sp['PV_Pts_Time']  = (sp['Final_EV_Pts'] * sp['Pct_Time']).where(P)              # planned points by this date
sp['Pct_Done']     = (sp['Cum_EV_Pts'] / sp['Final_EV_Pts']).where(P)

sp['SPI'] = (sp['Cum_EV_Pts'] / sp['PV_Pts_Time']).where(P & (sp['Pct_Time'] >= MIN_TIME))   # = Pct_Done / Pct_Time
sp['CPI'] = (sp['Cum_EV_Hours'] / sp['Cum_AC_Hours']).where(P)
sp['Pct_Complete']    = (sp['Cum_EV_Pts'] / sp['Cum_BAC_Pts']).where(P)       # EV / BAC
sp['Pct_Complete_ND'] = (sp['Cum_EV_Pts'] / sp['Cum_BAC_Pts_ND']).where(P)
sp = sp.replace([np.inf, -np.inf], np.nan)
sp['Year'] = sp['Sprint_End'].dt.year
sp['Ideal_Remaining_Pts'] = (sp.groupby('Project_Name')['Cum_BAC_Pts'].transform('max') * (1 - sp['Pct_Time'])).where(sp['Priced'])
sp.to_csv('tawos_data/evm_sprint.csv', index=False)

ok = sp[sp['Priced']]
print(sp.groupby('Project_Name')['Sprint_Status'].value_counts().unstack(fill_value=0), "\n")
print(sp.groupby('Project_Name')[['Scope_Start_Pts', 'Scope_End_Pts']].sum().round(0), "\n")
print(ok.groupby(['Project_Name', 'Year'])[['Baseline_HPP', 'PV_Pts_Time', 'Cum_EV_Pts', 'BAC_Hours', 'Cum_EV_Hours',
                                            'Cum_AC_Hours', 'SPI', 'CPI', 'Pct_Complete', 'Pct_Complete_ND']].last().round(2), "\n")
print("Last sprint of each project (SPI should be 1.0):")
print(ok.groupby('Project_Name').tail(1)[['Project_Name', 'Sprint_End', 'Pct_Time', 'Pct_Done', 'SPI']]
        .round({'Pct_Time': 2, 'Pct_Done': 2, 'SPI': 2}))
print("\nPlan end (last earned point) vs last sprint end:")
print(pd.DataFrame({'Plan_End': t1, 'Last_Sprint_End': sp.groupby('Project_Name')['Sprint_End'].max()}))

Finished after the project's last sprint (not counted): 1 of 5455
Sprint_Status             Measurable  No baseline yet
Project_Name                                         
Appcelerator Studio              141                3
Mule                             230               13
Titanium Mobile Platform         159                3 

                          Scope_Start_Pts  Scope_End_Pts
Project_Name                                            
Appcelerator Studio              4,023.00       5,833.00
Mule                             8,559.00      12,729.00
Titanium Mobile Platform         2,960.00       4,009.00 

                               Baseline_HPP  PV_Pts_Time  Cum_EV_Pts  \
Project_Name             Year                                          
Appcelerator Studio      2013          4.42       329.42    1,119.00   
                         2014          8.65     1,097.41    2,934.00   
                         2015         10.71     1,890.82    3,853.00   
               

In [ ]:
P = 'Project_Name'
pct = lambda s: (s * 100).round(1)

# ---------- 1. Sprint coverage: pointed issues that belong to a sprint ----------
pt = main[main['Story_Point'].notna()]
print("1a. % of pointed issues with a Sprint_ID:\n", pct(pt['Sprint_ID'].notna().groupby(pt[P]).mean()), "\n")
print("1b. % with a sprint and both sprint dates (what the sprint analysis sees):\n",
      pct((pt['Sprint_ID'].notna() & pt['Sprint_Start'].notna() & pt['Sprint_End'].notna()).groupby(pt[P]).mean()), "\n")



1a. % of pointed issues with a Sprint_ID:
 Project_Name
Appcelerator Studio        38.00
Mule                       81.50
Titanium Mobile Platform   76.10
Name: Sprint_ID, dtype: float64 

1b. % with a sprint and both sprint dates (what the sprint analysis sees):
 Project_Name
Appcelerator Studio        38.00
Mule                       77.00
Titanium Mobile Platform   76.00
dtype: float64 



In [ ]:
# ---------- 2. Scope growth: end vs start of the sprints ----------
t = sp.groupby(P)[['Scope_Start_Pts', 'Scope_End_Pts']].sum()
print("2. Scope at sprint end vs start (% larger):\n", pct(t['Scope_End_Pts'] / t['Scope_Start_Pts'] - 1), "\n")


2. Scope at sprint end vs start (% larger):
 Project_Name
Appcelerator Studio        45.00
Mule                       48.70
Titanium Mobile Platform   35.40
dtype: float64 



In [ ]:
# ---------- 3. Completed issues: points at sprint start, sprint end and resolution ----------
c = iss[iss['Completed']].copy()
c['Pts_Res'] = scope_at(events_base, c, 'Resolution_Date')          # points on the day it was resolved
print("3a. % of completed points not in scope at sprint start:\n",
      pct(c[c['Pts_Start'] == 0].groupby(P)['Pts_Res'].sum() / c.groupby(P)['Pts_Res'].sum()), "\n")
print("3b. % of completed issues with more points at resolution than at sprint end:\n",
      pct((c['Pts_Res'] > c['Pts_End_ND']).groupby(c[P]).mean()), "\n")
print("3c. % of completed issues with zero points at sprint end:\n",
      pct((c['Pts_End_ND'] == 0).groupby(c[P]).mean()), "\n")



3a. % of completed points not in scope at sprint start:
 Project_Name
Appcelerator Studio        38.70
Mule                       39.60
Titanium Mobile Platform   37.80
Name: Pts_Res, dtype: float64 

3b. % of completed issues with more points at resolution than at sprint end:
 Project_Name
Appcelerator Studio         1.70
Mule                        2.80
Titanium Mobile Platform   10.40
dtype: float64 

3c. % of completed issues with zero points at sprint end:
 Project_Name
Appcelerator Studio         3.30
Mule                       12.90
Titanium Mobile Platform   11.00
Name: Pts_End_ND, dtype: float64 



In [ ]:
# ---------- 4. Sprints that feed the cumulative columns: BAC vs EV, per outcome ----------
pr = iss.merge(sp[[P, 'Sprint_ID', 'Priced']], on=[P, 'Sprint_ID'], how='left')
pr = pr[pr['Priced'] == True].copy()
pr['Outcome'] = np.select([pr['Completed'], pr['Closed_No_Delivery'], pr['Finished']],
                          ['Completed', 'Closed without delivery', 'Other finished'], default='Not finished')
pr['Pts_Res'] = scope_at(events_base, pr, 'Resolution_Date').where(pr['Completed'], 0)
rec = pr.groupby([P, 'Outcome']).agg(Issues=('ID', 'count'),
                                      Pts_At_Sprint_End=('Pts_End_ND', 'sum'),
                                      Pts_At_Resolution=('Pts_Res', 'sum')).round(0)
print("4a. Reconciliation (completed rows: points in scope vs points earned):\n", rec, "\n")
cc = rec.xs('Completed', level='Outcome')
print("4b. Earned points vs points in scope, completed issues (% gap):\n",
      pct(cc['Pts_At_Resolution'] / cc['Pts_At_Sprint_End'] - 1), "\n")



4a. Reconciliation (completed rows: points in scope vs points earned):
                                                   Issues  Pts_At_Sprint_End  \
Project_Name             Outcome                                              
Appcelerator Studio      Closed without delivery     163             781.00   
                         Completed                  1074           5,371.00   
                         Not finished                 24             130.00   
                         Other finished                1               5.00   
Mule                     Closed without delivery     211             847.00   
                         Completed                  2954          12,281.00   
                         Not finished                  8              19.00   
Titanium Mobile Platform Closed without delivery      59             384.00   
                         Completed                   886           3,629.00   
                         Not finished                  3   

In [ ]:
# ---------- 5. Completed issues that never had points ----------
done = pr[pr['Completed']]
zero_both = (done['Pts_End_ND'] == 0) & (done['Pts_Res'] == 0)
print("5a. % of completed issues with no points at sprint end AND at resolution:\n",
      pct(zero_both.groupby(done[P]).mean()), "\n")
z = done[zero_both].merge(main[['ID', 'Estimation_Date']], on='ID')
print("5b. ...of which estimated after being resolved:\n",
      pct((z['Estimation_Date'] > z['Resolution_Date']).groupby(z[P]).mean()), "\n")
num = done[zero_both].groupby(P)['Capped_Hours'].sum()
print("5c. % of the hours of completed issues held by them:\n",
      pct(num / done.groupby(P)['Capped_Hours'].sum()), "\n")
print("5d. % of the hours of all finished issues held by them:\n",
      pct(num / pr[pr['Finished']].groupby(P)['Capped_Hours'].sum()), "\n")



5a. % of completed issues with no points at sprint end AND at resolution:
 Project_Name
Appcelerator Studio         1.60
Mule                       10.90
Titanium Mobile Platform    0.80
dtype: float64 

5b. ...of which estimated after being resolved:
 Project_Name
Appcelerator Studio        0.00
Mule                       0.00
Titanium Mobile Platform   0.00
dtype: float64 

5c. % of the hours of completed issues held by them:
 Project_Name
Appcelerator Studio        1.60
Mule                       6.30
Titanium Mobile Platform   0.90
Name: Capped_Hours, dtype: float64 

5d. % of the hours of all finished issues held by them:
 Project_Name
Appcelerator Studio        1.40
Mule                       6.10
Titanium Mobile Platform   0.90
Name: Capped_Hours, dtype: float64 



In [ ]:
# ---------- 6. How did the delivered points enter after the sprint started? ----------
add = done[done['Pts_Start'] == 0].merge(main[['ID', 'Creation_Date', 'Estimation_Date', 'Type', 'Title']], on='ID')
add['How_Added'] = np.select(
    [add['Creation_Date'] > add['Sprint_Start'], add['Estimation_Date'] > add['Sprint_Start']],
    ['New issue (created after start)', 'Existing issue (estimated after start)'],
    default='No points history')
print("6a. Share of the added points, by how they entered:\n",
      pd.crosstab(add[P], add['How_Added'], values=add['Pts_Res'], aggfunc='sum', normalize='index').round(2), "\n")
print("6b. Share of the added points, by issue type:\n",
      pd.crosstab(add[P], add['Type'], values=add['Pts_Res'], aggfunc='sum', normalize='index').round(2).to_string(), "\n")



6a. Share of the added points, by how they entered:
 How_Added                 New issue (created after start)  No points history
Project_Name                                                                
Appcelerator Studio                                  0.63               0.37
Mule                                                 0.70               0.30
Titanium Mobile Platform                             0.43               0.57 

6b. Share of the added points, by issue type:
 Type                      Bug  Enhancement Request  Epic  Improvement  New Feature  Story  Sub-task  Task  Technical task
Project_Name                                                                                                             
Appcelerator Studio      0.62                 0.00  0.00         0.18         0.05   0.13      0.02  0.00            0.00
Mule                     0.37                 0.23  0.00         0.00         0.00   0.01      0.00  0.38            0.00
Titanium Mobile Platform 

In [ ]:
# ---------- 7. Technical-debt proxy (rough: Technical task type or keywords in the title) ----------
words = r'refactor|tech(?:nical)? debt|clean ?up|deprecat|legacy|migrat'
d = done.merge(main[['ID', 'Type', 'Title']], on='ID')
d['Debt_Proxy'] = (d['Type'] == 'Technical task') | d['Title'].str.contains(words, case=False, na=False)
d['Added_After_Start'] = d['Pts_Start'] == 0
print("7a. % of completed issues flagged as technical debt:\n", pct(d.groupby(P)['Debt_Proxy'].mean()), "\n")
print("7b. ...split by added after the sprint started or not:\n",
      pct(d.groupby([P, 'Added_After_Start'])['Debt_Proxy'].mean()))

7a. % of completed issues flagged as technical debt:
 Project_Name
Appcelerator Studio        5.20
Mule                       4.60
Titanium Mobile Platform   2.80
Name: Debt_Proxy, dtype: float64 

7b. ...split by added after the sprint started or not:
 Project_Name              Added_After_Start
Appcelerator Studio       False               8.30
                          True                1.30
Mule                      False               5.70
                          True                3.70
Titanium Mobile Platform  False               2.90
                          True                2.80
Name: Debt_Proxy, dtype: float64


In [ ]:
# ---------- 8. What are the extra points added after each sprint started? ----------
x = iss.copy()                                      # every pointed issue that belongs to a sprint
meta = main.set_index('ID')[['Type', 'Creation_Date']]
x['Type']    = x['ID'].map(meta['Type'])
x['Created'] = pd.to_datetime(x['ID'].map(meta['Creation_Date'])).astype(ns)

x['Delta']   = x['Pts_End'] - x['Pts_Start']        # points gained (or lost) between sprint start and end
x['Added']   = x['Delta'].clip(lower=0)
x['Removed'] = (-x['Delta']).clip(lower=0)
x['Origin']  = np.select(
    [x['Delta'] <= 0, x['Created'] > x['Sprint_Start'], x['Pts_Start'] == 0],
    ['(no added points)', 'New issue created after sprint start', 'Existing issue, had no points before the sprint'],
    default='Existing issue, points raised')

# 8a. Check: added - removed must equal scope end - scope start from the sprint table
chk = x.groupby('Project_Name')[['Added', 'Removed', 'Delta']].sum()
chk['Scope_End - Start (sprint table)'] = (sp.groupby('Project_Name')['Scope_End_Pts'].sum()
                                           - sp.groupby('Project_Name')['Scope_Start_Pts'].sum())
print("8a. Added / removed points and reconciliation:\n", chk.round(0), "\n")

a = x[x['Added'] > 0]

# 8b. Which issue types hold the added points (% of all added points per project)
by_type = a.pivot_table(index='Type', columns='Project_Name', values='Added', aggfunc='sum', fill_value=0)
by_type = by_type.loc[by_type.sum(axis=1).sort_values(ascending=False).index]
print("8b. % of added points by issue type:\n", (by_type / by_type.sum() * 100).round(1).head(8), "\n")

# 8c. Where the added points come from
by_origin = a.pivot_table(index='Origin', columns='Project_Name', values='Added', aggfunc='sum', fill_value=0)
print("8c. % of added points by origin:\n", (by_origin / by_origin.sum() * 100).round(1), "\n")

# 8d. Type x origin, per project (points)
print("8d. Added points by origin and type:\n",
      a.pivot_table(index=['Project_Name', 'Origin'], columns='Type', values='Added', aggfunc='sum', fill_value=0)
        .round(0).loc[:, lambda d: d.sum().sort_values(ascending=False).index[:6]], "\n")

# 8e. Average extra points per sprint
n_sprints = sp.groupby('Project_Name').size()
print("8e. Average points added per sprint:\n", (chk['Added'] / n_sprints).round(1))

8a. Added / removed points and reconciliation:
                             Added  Removed    Delta  \
Project_Name                                          
Appcelerator Studio      2,236.00   426.00 1,810.00   
Mule                     5,011.00   841.00 4,170.00   
Titanium Mobile Platform 1,229.00   180.00 1,049.00   

                          Scope_End - Start (sprint table)  
Project_Name                                                
Appcelerator Studio                               1,810.00  
Mule                                              4,170.00  
Titanium Mobile Platform                          1,049.00   

8b. % of added points by issue type:
 Project_Name         Appcelerator Studio  Mule  Titanium Mobile Platform
Type                                                                    
Bug                                62.60 36.90                     25.10
Task                                0.00 38.20                      0.00
Enhancement Request                 0.0

## Per-sprint earned value: PV, EV, AC, BAC, SPI and CPI

### Method

Earned value is calculated **per sprint**, because sprints are the project's own unit of planning and delivery. Everything is measured in story points and converted to hours with a baseline, so the figures stay comparable across sprints with different amounts of work.

| Measure | How it is calculated |
|---|---|
| **Baseline hours per point** | Total capped hours ÷ total completed points of **all previous sprints** (needs at least 100 completed points; earlier sprints are marked "no baseline" and left out of the EVM) |
| **EV** (earned value) | Points of issues completed in the sprint (Fixed / Done / Completed) × baseline |
| **AC** (actual cost, in hours) | Capped hours of the issues finished in the sprint |
| **BAC** (budget at completion) | Cumulative scope at the **end** of each sprint (minus issues closed without delivery) × baseline |
| **PV** (planned value) | Final earned points × share of project time elapsed (linear, time-based plan) |
| **SPI** | Cumulative EV points ÷ cumulative PV points (= % done ÷ % of time elapsed) |
| **CPI** | Cumulative EV hours ÷ cumulative AC hours |
| **% complete** | Cumulative EV points ÷ cumulative BAC points |

The scope of each sprint is rebuilt **as of a date** (sprint start or sprint end) from the STORY_POINT change log, not from the final value of the field. Otherwise an issue re-estimated months later would be counted with a number nobody knew at that moment.

### Why PV is time-based

My first version used the points in scope **at the start of each sprint** as the plan. That gave SPI values of 1.28 (Appcelerator), 1.50 (Mule) and 1.33 (Titanium) at the end of the projects, which looks like "ahead of schedule" but isn't. The cause is that scope kept growing **inside** the sprints: by the end of the sprint the scope was +45 % (Appcelerator), +49 % (Mule) and +35 % (Titanium) bigger than at the start. Work that was never planned was counted as "earned beyond plan".

The time-based version avoids this. Planned value is a straight line from the first priced sprint to the last sprint end: if 40 % of the project time has passed, the plan says 40 % of the final points should be done. SPI is then simply:

> SPI = % of work done ÷ % of time elapsed

SPI > 1 means the project is further along than the calendar suggests, SPI < 1 means it is behind.

### Why SPI ends at 1.0, and why a value above 1 does not mean "ahead of schedule"

**It ends at 1.0 by construction.** At the last sprint, 100 % of the time has elapsed and 100 % of the final earned points are done, so SPI = 1 / 1 = 1.0 for every project. It is not a result, it is a property of the formula (the same happens in textbook EVM when a project finishes). So:

- The **final** SPI says nothing. The **trajectory** is what is informative: where the project was above or below an even pace on the way there.
- The first 5 % of the timeline is left blank, because dividing by a very small share of time produces extreme values.

**Why a value above 1 is not proof of being ahead of schedule.** In a real project, SPI compares progress with a plan that was agreed *before* the work started. TAWOS has no such plan, so there are three reasons to be careful:

1. **The plan is an assumption.** The planned value is a straight line: if 40 % of the time has passed, 40 % of the work should be done. Real projects are rarely evenly paced. Appcelerator and Titanium delivered most of their points early, so against a straight line they look 2–3× "ahead" for years, until the line catches up. That describes the shape of the delivery, not a team that beat a schedule.
2. **The target kept moving.** Scope grew after the sprints started (+45 %, +49 % and +35 % in Appcelerator, Mule and Titanium), mostly through new bugs, requests and previously unestimated work (see the section on why scope changes). With a target that grows while the sprint is running, "ahead of plan" has no fixed meaning. My first version made this problem visible: it used the points in scope at the start of each sprint as the plan, and the scope growth alone pushed SPI to 1.28 / 1.50 / 1.33 at the end of the projects. That version was dropped because it rewarded growth, not speed.
3. **The final total is only known in hindsight.** The plan uses the total points that were eventually delivered, which nobody knew at the time.

**How to read it:**

| SPI | What it means here | What it does **not** mean |
|---|---|---|
| Above 1 | More of the final work was done than an even pace would give at that date | The team beat a schedule |
| Around 1 | Delivery followed an even pace | Delivery was on plan |
| Below 1 | Delivery came later than an even pace (Mule in 2014–2015) | The team was late against a commitment |

The SPI is best used to compare the *shape* of delivery between projects and over time. For "did the sprints deliver what they held?", use the completion view (delivered ÷ committed points, 0.85 / 0.95 / 0.98 at the end), which doesn't depend on a straight-line plan.

### What the results show

- **SPI starts high and converges to 1.0.** Appcelerator goes from 3.44 to 1.00 and Titanium from 2.81 to 1.00. Early on a small amount of work done in a very short elapsed time looks fast, and the ratio settles as the project matures.
- **Mule is the only project that dips below 1.** It sits at 0.70 and 0.62 in the early years (behind a straight-line plan), recovers to 0.99, peaks at 1.38 and then comes back down 1.30 → 1.21 → 1.00. Most of its work was delivered in the middle years rather than evenly.
- **% complete at the end is 0.97 (Appcelerator), 1.02 (Mule) and 1.09 (Titanium).** Close to 100 % is expected; the deviations are explained in the limits below.
- **CPI ends at 0.43 (Appcelerator), 0.97 (Mule) and 0.61 (Titanium).** Mule's CPI is the most stable (0.96–0.99 from 2017). Appcelerator and Titanium spent more hours per point than the baseline predicted, so their cost is well above the earned value. BAC is built from the same baseline, so a low CPI also means the budget derived from it would have been exceeded.
- **Sprints measurable:** 141 (Appcelerator), 230 (Mule) and 159 (Titanium). Only 3, 13 and 3 sprints respectively had no baseline yet.

### Why scope changes after the sprint starts

Scope at the end of the sprint is much bigger than at the start (5,833 vs 4,023 points in Appcelerator, 12,729 vs 8,559 in Mule and 4,009 vs 2,960 in Titanium). About 38–40 % of the completed points were not in scope when their sprint started (38.7 % / 39.6 % / 37.8 %).

The net growth hides two movements. Sprints **gained** 2,236 / 5,011 / 1,229 points after they started and **lost** 426 / 841 / 180 (issues dropped or re-estimated downwards), which gives the net +1,810 / +4,170 / +1,049. On average a sprint gains about **15.5, 20.6 and 7.6 points** (Appcelerator, Mule, Titanium) after it has started. The rest of this section looks at where those added points come from.

#### Where the added points come from

| Origin of the added points | Appcelerator | Mule | Titanium |
|---|---|---|---|
| **New issue** created after the sprint started | 62 % | 68 % | 47 % |
| **Existing issue that had no points** before the sprint (known work, estimated only when pulled in) | 37 % | 29 % | 49 % |
| **Existing issue whose points were raised** | 1 % | 3 % | 4 % |

Most of the growth is new work arriving in the middle of a sprint. In Titanium, almost half comes from work that already existed but had never been estimated. Re-estimating issues that already had points is minor everywhere.

#### What kind of work it is

| Issue type | Appcelerator | Mule | Titanium |
|---|---|---|---|
| Bug | 62.6 % | 36.9 % | 25.1 % |
| Task | – | 38.2 % | – |
| Enhancement Request | – | 23.7 % | – |
| Improvement | 18.2 % | – | 30.3 % |
| New Feature | 2.4 % | – | 28.9 % |
| Story | 13.6 % | 1.2 % | 13.6 % |
| Sub-task | 2.3 % | – | 1.4 % |
| Technical task | 0.9 % | – | 0.7 % |

(– means the project has no issues of that type, or none with added points.)

- **Appcelerator:** the added points are mostly **bugs** (63 %), and two thirds of the points from new issues are bugs. It is reactive work: problems found while the sprint is running.
- **Mule:** three types share it: **tasks** (38 %), **bugs** (37 %) and **enhancement requests** (24 %). Bugs and tasks are mainly new issues (bugs: 1,421 of 1,849 points came from new issues), so it is both reactive work and requests that arrive during the sprint.
- **Titanium:** the mix is the broadest: **improvements** (30 %), **new features** (29 %), **bugs** (25 %) and **stories** (14 %). New issues are mostly new features and improvements (260 and 171 of 583 points), while previously unestimated work is mostly bugs and improvements. In other words, Titanium's team was adding new product work during sprints, not just fixing problems.

#### Is the extra scope technical debt? (the "debt proxy")

A natural hypothesis is that unplanned work is cleanup of earlier shortcuts: refactoring, migrations, deprecations. TAWOS has no field that says "this is technical debt", so I used two checks.

**Check 1: the issue type.** `Technical task` is the closest type to debt, and it holds under **1 %** of the added points (0.9 % in Appcelerator and 0.7 % in Titanium; Mule has no such type). It cannot explain the growth.

**Check 2: a rough proxy.** I flagged each completed issue as debt-related if:

- its type is **Technical task**, or
- its title contains one of: *refactor*, *tech debt / technical debt*, *cleanup / clean up*, *deprecat…*, *legacy*, *migrat…*

Then I compared the share of flagged issues among work that was in scope at the start of the sprint with work added after it started:

| Project | All completed issues | In scope at start | Added after start |
|---|---|---|---|
| Appcelerator Studio | 5.2 % | 8.3 % | 1.3 % |
| Mule | 4.6 % | 5.7 % | 3.7 % |
| Titanium Mobile Platform | 2.8 % | 2.9 % | 2.8 % |

If debt were driving the extra scope, the "added after start" column would be higher than the "in scope at start" column. It is lower or equal in all three projects: debt-related work is planned more often than it is added. (Check 2 counts issues, check 1 counts points, so the percentages are not directly comparable, but they point the same way.)

**Conclusion:** technical debt does not explain the in-sprint growth. It is mostly new bugs and requests that arrive during the sprint, plus known work that was not estimated until it was pulled in.

**How far to trust the proxy:** it only catches issues that say so in their type or title. Debt work described in other words is missed (false negatives), and a title like "migrate user to new API" may be a feature, not debt (false positives). Mule has no Technical task type, so for Mule the proxy relies on titles only. I use it as a directional test, not as a measurement of debt.

### Limits

- **Straight-line plan and hindsight.** The planned value assumes work should be spread evenly over time, and the final points (and the project end date) are only known afterwards. The SPI is useful to compare the shape of the trajectories, not as a forecast that could have been calculated live.
- **Last-sprint assignment.** An issue that moved between sprints is booked in the last sprint it belonged to when it was resolved, not in the one where it started.
- **Titanium goes above 100 % (1.06–1.09).** About 10 % of its completed issues gained points between the end of the sprint and their resolution, so it earned 3,946 points against a scope of 3,629 (+8.7 %). Mule is +2.0 % and Appcelerator is −1.5 %.
- **Appcelerator is only partially covered.** Only 38 % of its pointed issues belong to a sprint, against 81.5 % in Mule and 76 % in Titanium, so the EVM covers about a third of the project's estimated work. For the full picture of cost and hours, use the overview (sections 1.1 and 1.4).
- **Issues with no story points.** They are 11 % of Mule's completed issues and 1–2 % in the others, and hold 6 % of Mule's hours (1–2 % elsewhere). They are left out of EV but their hours are included in AC, so the effect on CPI is small. A check for "estimated after being resolved" finds none, but it is a weak check, because the change log may simply not record it.
- **Effort is elapsed time, capped at 720 h per issue.** Hours are not timesheets, so cost and CPI are upper bounds, not exact figures.

# Synthetic Budget recreation #

In [ ]:

need = ['Capped_Hours', 'Capped_Cost_USD', 'Actual_Cost_USD', 'Hourly_Rate_USD', 'Rate_Source', 'Seniority_Tier']
assert all(c in main.columns for c in need), [c for c in need if c not in main.columns]

# ---------- 1. Rates by tier (the assumption you can change) ----------
rate_bands = {'Junior': (15, 20), 'Junior-Mid': (20, 26), 'Mid': (26, 33),
              'Mid-Senior': (33, 42), 'Senior': (42, 55)}                   # USD/hour
team = pd.read_csv('tawos_data/team_profiles_v1.csv')
tiers = (team.groupby('Seniority_Tier')['Hourly_Rate_USD']
             .agg(People='count', Mean_Rate='mean').reindex(list(rate_bands)))
tiers['Band_USD_h'] = [f"{lo}-{hi}" for lo, hi in rate_bands.values()]
print(tiers.round(2), "\n")

# ---------- 2. Where does the cost come from? ----------
main['Tier_Label'] = main['Seniority_Tier'].fillna('No tier (assumed rate)')
by_tier = (main.groupby('Tier_Label')
               .agg(Issues=('ID', 'count'), Hours=('Capped_Hours', 'sum'), Cost_USD=('Capped_Cost_USD', 'sum')))
by_tier['Pct_of_Cost'] = (by_tier['Cost_USD'] / by_tier['Cost_USD'].sum() * 100).round(1)
print(by_tier.round(0), "\n")

# ---------- 3. Actual cost per project (capped hours x rate) ----------
cost = main.groupby('Project_Name').agg(Hours=('Capped_Hours', 'sum'),
                                        Actual_Cost_USD=('Capped_Cost_USD', 'sum'),
                                        Uncapped_Cost_USD=('Actual_Cost_USD', 'sum'))
assumed = main[main['Rate_Source'].fillna('').str.startswith('Assumed')].groupby('Project_Name')['Capped_Cost_USD'].sum()
cost['Pct_Cost_Assumed_Rate'] = (assumed / cost['Actual_Cost_USD'] * 100).round(1)
years = main.groupby('Project_Name')['Creation_Date'].agg(lambda s: (s.max() - s.min()).days / 365)

# ---------- 4. The three simulated situations ----------
RULES = {'Mule':                     {'scenario': 'Under budget (comfortable)', 'factor': 1.20},
         'Titanium Mobile Platform': {'scenario': 'Over budget (tight)',        'factor': 0.85},
         'Appcelerator Studio':      {'scenario': 'On budget (fixed annual)',   'factor': None}}

rows = []
for p, r in RULES.items():
    actual = cost.loc[p, 'Actual_Cost_USD']
    if r['factor']:
        budget, annual = actual * r['factor'], np.nan
    else:                                                    # fixed yearly amount, rounded to $100K
        annual = round((actual / years[p]) / 100_000) * 100_000
        budget = annual * years[p]
    rows.append({'Project_Name': p, 'Scenario': r['scenario'], 'Factor': r['factor'],
                 'Annual_Budget_USD': annual, 'Simulated_Budget_USD': budget})
budgets = cost.reset_index().merge(pd.DataFrame(rows), on='Project_Name')
budgets['Variance_USD'] = budgets['Simulated_Budget_USD'] - budgets['Actual_Cost_USD']   # + = under budget
budgets['Variance_%'] = (budgets['Variance_USD'] / budgets['Simulated_Budget_USD'] * 100).round(1)
budgets['Budget_Status'] = np.select([budgets['Variance_%'] > 5, budgets['Variance_%'] < -5],
                                     ['Under budget', 'Over budget'], default='On budget (within 5%)')
budgets.to_csv('tawos_data/project_budgets.csv', index=False)

show = budgets.set_index('Project_Name')
print((show[['Actual_Cost_USD', 'Uncapped_Cost_USD', 'Simulated_Budget_USD', 'Variance_USD']] / 1e6).round(2))
print(show[['Scenario', 'Variance_%', 'Budget_Status', 'Pct_Cost_Assumed_Rate']])

# ---------- 5. Monthly actual cost (for Tableau) ----------
monthly = (main.dropna(subset=['Resolution_Date'])
               .assign(Month=lambda d: d['Resolution_Date'].dt.to_period('M').dt.to_timestamp())
               .groupby(['Project_Name', 'Month'])['Capped_Cost_USD'].sum().reset_index())
monthly['Cum_Actual_Cost_USD'] = monthly.groupby('Project_Name')['Capped_Cost_USD'].cumsum()
monthly.to_csv('tawos_data/cost_monthly.csv', index=False)

                People  Mean_Rate Band_USD_h
Seniority_Tier                              
Junior              88      18.81      15-20
Junior-Mid          21      23.50      20-26
Mid                 34      29.76      26-33
Mid-Senior          19      37.78      33-42
Senior              20      45.91      42-55 

                        Issues      Hours      Cost_USD  Pct_of_Cost
Tier_Label                                                          
Junior                     956  97,170.00  1,819,660.00         4.00
Junior-Mid                1081  71,409.00  1,698,566.00         4.00
Mid                       2264 126,609.00  3,860,716.00         9.00
Mid-Senior                3372 254,563.00  9,896,956.00        24.00
No tier (assumed rate)    1959  27,310.00    857,271.00         2.00
Senior                   11222 484,634.00 22,902,660.00        56.00 

                          Actual_Cost_USD  Uncapped_Cost_USD  \
Project_Name                                                   
A

## Rates, cost in dollars and simulated budgets

### Hourly rates by seniority tier
Each of the 182 contributors has a tier and an hourly rate inside the tier's band. The position inside the band
follows the contributor's seniority score. **The bands are an assumption** (USD per hour):

| Tier | People | Band | Mean rate |
|---|---|---|---|
| Junior | 88 | 15-20 | 18.81 |
| Junior-Mid | 21 | 20-26 | 23.50 |
| Mid | 34 | 26-33 | 29.76 |
| Mid-Senior | 19 | 33-42 | 37.78 |
| Senior | 20 | 42-55 | 45.91 |

Issues with no tiered owner (1,959, about 2% of cost) are priced at the core-team median rate ($31.39/h) and labelled
as assumed in `Rate_Source`.

### Actual cost
Cost = capped effort hours × the owner's hourly rate. Total **$41.0M**.

| Tier | Issues | Hours | Cost | Share of cost |
|---|---|---|---|---|
| Junior | 956 | 97,170 | $1.82M | 4% |
| Junior-Mid | 1,081 | 71,409 | $1.70M | 4% |
| Mid | 2,264 | 126,609 | $3.86M | 9% |
| Mid-Senior | 3,372 | 254,563 | $9.90M | 24% |
| Senior | 11,222 | 484,634 | $22.90M | 56% |
| No tier (assumed rate) | 1,959 | 27,310 | $0.86M | 2% |

The Senior share is largely a result of how tiers were built: issue volume is part of the seniority score, so the
contributors with the most issues are Senior by construction.

### Simulated budgets
TAWOS has no real budgets, so three situations are simulated from each project's actual cost:

| Project | Rule | Actual | Budget | Variance |
|---|---|---|---|---|
| Mule | Actual × 1.20 (comfortable) | $22.05M | $26.46M | +$4.41M (+16.7%) |
| Titanium | Actual × 0.85 (tight) | $10.93M | $9.29M | −$1.64M (−17.6%) |
| Appcelerator | Fixed annual amount rounded to $100K × years | $8.06M | $7.72M | −$0.34M (−4.4%) |

**The variances are built in by construction.** Mule and Titanium come straight from the factors, and Appcelerator's
small gap comes from rounding the annual amount. They show how the analysis behaves in three situations, not how well
these projects were run. Budgets follow actual cost, so they move whenever the cost changes.

### Effect of the effort cap
Costs use effort capped at 720 hours per ticket. Without the cap they would be much higher (Appcelerator $13.9M,
Mule $47.5M, Titanium $31.3M), because a few hundred tickets carry most of the recorded hours (see the flagged
tickets above). The budgets are built on the capped cost.

### Output
`project_budgets.csv` and `cost_monthly.csv`. Column `Simulated_Budget_USD` is not the same as BAC in the sprint analysis
(BAC is scope in hours, and this is a simulated dollar budget).

In [ ]:
# ---------- 1. Checks before saving ----------
print("Rows:", len(main), "(expected 20,854) | unique IDs:", main['ID'].is_unique)
dups = [c for c in main.columns if c.endswith(('_x', '_y'))]
print("Leftover _x/_y columns:", dups)
print("Duplicated column names:", main.columns[main.columns.duplicated()].tolist())

key = ['Capped_Hours', 'Capped_Cost_USD', 'Hourly_Rate_USD', 'Rate_Source', 'Owner_Category',
       'Assignee_ID_Final', 'Owner_Source', 'Is_Effort_Outlier', 'Seniority_Tier', 'Tier_Confidence']
print(main[[c for c in key if c in main.columns]].isna().mean().round(3).to_string())   # share of blanks per column
print("Issues with no rate:", main['Hourly_Rate_USD'].isna().sum())

# ---------- 2. Save (the two long text columns are dropped so Tableau loads faster) ----------
out = main.drop(columns=['Description', 'Description_Text'], errors='ignore')
out.to_csv('tawos_data/projects_dataset.csv', index=False)
print("Saved projects_dataset.csv:", out.shape)

# ---------- 3. What is in the folder now ----------
for f in ['projects_dataset', 'evm_monthly', 'evm_sprint', 'project_budgets', 'cost_monthly', 'team_profiles_v1']:
    path = f'tawos_data/{f}.csv'
    print(f"{f}.csv:", pd.read_csv(path, nrows=0).shape[1], "columns" if os.path.exists(path) else "-> MISSING")

Rows: 20854 (expected 20,854) | unique IDs: True
Leftover _x/_y columns: []
Duplicated column names: []
Capped_Hours        0.00
Capped_Cost_USD     0.00
Hourly_Rate_USD     0.00
Rate_Source         0.00
Owner_Category      0.00
Assignee_ID_Final   0.08
Owner_Source        0.08
Is_Effort_Outlier   0.00
Seniority_Tier      0.09
Tier_Confidence     0.09
Issues with no rate: 0
Saved projects_dataset.csv: (20854, 67)
projects_dataset.csv: 67 columns
evm_monthly.csv: 19 columns
evm_sprint.csv: 33 columns
project_budgets.csv: 12 columns
cost_monthly.csv: 4 columns
team_profiles_v1.csv: 70 columns


## evm calculation with these simulated budgets

In [ ]:
# ---------- 9. EVM in USD using the simulated budgets ----------
pb = pd.read_csv('tawos_data/project_budgets.csv').set_index('Project_Name')
MIN_DONE = 0.10                                   # EAC / VAC are blank until 10 % of the points are delivered
pbi = pb.set_index('Project_Name') if 'Project_Name' in pb.columns else pb
BAC_TOTAL = pbi['Simulated_Budget_USD']

# Cost of every resolved issue (capped hours x hourly rate), by resolution date
c = main[main['Status'].isin(RESOLVED) & main['Resolution_Date'].notna()][
    ['Project_Name', 'Resolution_Date', 'Capped_Hours', 'Hourly_Rate_USD']].copy()
c['Resolution_Date'] = c['Resolution_Date'].astype(ns)
print("Resolved issues with no hourly rate (should be 0):", int(c['Hourly_Rate_USD'].isna().sum()))
c['Cost'] = c['Capped_Hours'] * c['Hourly_Rate_USD']
c = c.sort_values('Resolution_Date')

# Whole-project cumulative cost at each sprint end
sp['AC_All_USD'] = np.nan
for p, idx in sp.groupby('Project_Name').groups.items():
    g = c[c['Project_Name'] == p]
    cs = g['Cost'].cumsum().values
    pos = np.searchsorted(g['Resolution_Date'].values, sp.loc[idx, 'Sprint_End'].values, side='right')
    sp.loc[idx, 'AC_All_USD'] = np.where(pos > 0, cs[np.maximum(pos - 1, 0)], 0.0)

P = sp['Priced']
# The cost clock starts at the first priced sprint, like the hours-based CPI
t0 = sp[P].groupby('Project_Name')['Sprint_Start'].min()
AC_pre  = pd.Series({p: c.loc[(c['Project_Name'] == p) & (c['Resolution_Date'] <= t0[p]), 'Cost'].sum() for p in t0.index})
AC_last = sp[P].groupby('Project_Name')['AC_All_USD'].last()
BAC_win = BAC_TOTAL * (AC_last - AC_pre) / AC_last        # budget share for the measured window (proportional to cost)
print("Cost before the first priced sprint (left out of CPI):\n", pd.DataFrame({'AC_pre': AC_pre, 'Share_of_last_AC': AC_pre / AC_last}).round(2), "\n")

sp['BAC_Total_USD'] = sp['Project_Name'].map(BAC_TOTAL)
sp['BAC_USD'] = sp['Project_Name'].map(BAC_win)
sp['AC_USD']  = (sp['AC_All_USD'] - sp['Project_Name'].map(AC_pre)).where(P)
sp['PV_USD']  = (sp['BAC_USD'] * sp['Pct_Time']).where(P)
sp['EV_USD']  = (sp['BAC_USD'] * sp['Pct_Done']).where(P)
sp['CPI_USD'] = (sp['EV_USD'] / sp['AC_USD']).where(P)
sp['Pct_Budget_Used'] = (sp['AC_All_USD'] / sp['BAC_Total_USD']).where(P)            # whole project
sp['EAC_USD'] = (sp['Project_Name'].map(AC_pre) + sp['BAC_USD'] / sp['CPI_USD']).where(P & (sp['Pct_Done'] >= MIN_DONE))
sp['VAC_USD'] = (sp['BAC_Total_USD'] - sp['EAC_USD'])
sp = sp.replace([np.inf, -np.inf], np.nan)
sp.to_csv('tawos_data/evm_sprint.csv', index=False)

ok = sp[sp['Priced']]
print(ok.groupby(['Project_Name', 'Year'])[['BAC_USD', 'PV_USD', 'EV_USD', 'AC_USD', 'CPI_USD',
                                            'Pct_Budget_Used', 'EAC_USD', 'VAC_USD']].last().round(2), "\n")
last = ok.groupby('Project_Name').tail(1).set_index('Project_Name')
print("Check: whole-project AC at the last sprint vs Actual_Cost_USD in project_budgets (gap = cost of issues still open)")
print(pd.DataFrame({'AC_All_last_sprint': last['AC_All_USD'], 'Budget_table_actual': pbi['Actual_Cost_USD'],
                    'Gap': pbi['Actual_Cost_USD'] - last['AC_All_USD']}).round(0))

Resolved issues with no hourly rate (should be 0): 0
Cost before the first priced sprint (left out of CPI):
                                AC_pre  Share_of_last_AC
Appcelerator Studio      1,808,157.48              0.23
Mule                     8,066,123.56              0.37
Titanium Mobile Platform 3,616,502.47              0.33 

                                    BAC_USD        PV_USD        EV_USD  \
Project_Name             Year                                             
Appcelerator Studio      2013  5,908,544.31    365,661.28  1,242,093.01   
                         2014  5,908,544.31  1,218,125.70  3,256,747.89   
                         2015  5,908,544.31  2,098,820.69  4,276,840.36   
                         2016  5,908,544.31  2,974,239.41  4,794,101.61   
                         2017  5,908,544.31  3,848,462.08  5,416,813.12   
                         2018  5,908,544.31  4,687,033.66  5,806,424.06   
                         2019  5,908,544.31  5,526,694.52  5,857,